### Imports

In [2]:
from pathlib import Path
import json
import os 
import re

print("Imports Complete.")

Imports Complete.


### Inspect Directory

In [ ]:
current_directory = Path.cwd()               # returns the current working directory of this notebook
print("Current directory :")
print(current_directory)
print()
print("Files and Directories :")
for path in current_directory.iterdir():     # iterdir() will iterate over all the items in the directory
    print(path)

Current directory :
c:\Users\Shivanshu Sirohi\Documents\Multilingual_medical_BERT\stage_1_base_multilingual_model

Files and Directories :
c:\Users\Shivanshu Sirohi\Documents\Multilingual_medical_BERT\stage_1_base_multilingual_model\01_data_preparation.ipynb
c:\Users\Shivanshu Sirohi\Documents\Multilingual_medical_BERT\stage_1_base_multilingual_model\data
c:\Users\Shivanshu Sirohi\Documents\Multilingual_medical_BERT\stage_1_base_multilingual_model\README.md
c:\Users\Shivanshu Sirohi\Documents\Multilingual_medical_BERT\stage_1_base_multilingual_model\tokenization_fundamentals


In [9]:
data_directory = current_directory / 'data'
print("Data directory :")
print(data_directory)
print()
print("Files and Directories")
for path in data_directory.iterdir():
    print(" ", path.name)

Data directory :
c:\Users\Shivanshu Sirohi\Documents\Multilingual_medical_BERT\stage_1_base_multilingual_model\data

Files and Directories
  enwiki-2026-09-01-p80258756p84110898.jsonl.gz


In [11]:
corpus_path = data_directory / 'enwiki-2026-09-01-p80258756p84110898.jsonl.gz'
print("Corpus path :", corpus_path)
print("Exists      :", corpus_path.exists())
print("File name   :", corpus_path.name)
print("Suffixes    :", corpus_path.suffixes)

file_size_bytes = corpus_path.stat().st_size
file_size_mb = file_size_bytes / (1024 ** 2)

print(f"File size   : {file_size_mb:.2f} MB")

Corpus path : c:\Users\Shivanshu Sirohi\Documents\Multilingual_medical_BERT\stage_1_base_multilingual_model\data\enwiki-2026-09-01-p80258756p84110898.jsonl.gz
Exists      : True
File name   : enwiki-2026-09-01-p80258756p84110898.jsonl.gz
Suffixes    : ['.jsonl', '.gz']
File size   : 217.55 MB


In [13]:
import gzip                           # imports python library for reading/writing a .gz compressed file

with gzip.open(corpus_path, 'rt', encoding="utf-8") as file:       # open the compressed file directly in text reading mode
    first_line = file.readline()                                   # rt > r = read , t = text
                                                                   # readline reads only the first line/record from the corpus
    print("Raw line length :", len(first_line))
    print()
    print(first_line[:2000])



Raw line length : 1205

{"id":"enwiki_80258789","source":"wikipedia","wiki_id":"enwiki","snapshot":"2026-09-01","page_id":"80258789","title":"Ehsan Mehmood Khan","text":"is a retired two-star general of the Pakistan Army, who served as the Director General Infantry at the General Headquarters in Rawalpindi.\n\nEducation\nKhan has done his PhD in Peace and Conflict Studies.\n\nMilitary career\nKhan was commissioned into the 34 Punjab Regiment via the 22 OTS Course. In 2018, Khan was promoted to the rank Major General from Brigadier. As Force Commander Northern Areas, he oversaw military operations and civil-military initiatives in Gilgit-Baltistan. Later he served as Director General of the National Guard.\n\nHe also served as Director General Infantry at GHQ, Rawalpindi till retirement.\n\nHe was awarded the Hilal-i-Imtiaz by the then President of Pakistan in 2021 in recognition of his distinguished service.\n\nPublications\nKhan is the author of the book , which discusses non-traditio

In [14]:
## ^^ the text above is very well written and coherent. Exactly what we need to build our model

### Process JSON

In [18]:
first_record = json.loads(first_line)       # short for 'load strings' is used to parse a json formatted string and convert it into a corresponding python object
print("record type :", type(first_record))
print("fields      :", first_record.keys())
print()
for key, value in first_record.items():
    print(f"{key:10} | {type(value).__name__:8} | {repr(value)[:300]}")   # print key with a fixed width of 10 characters, value with a simple name like str instead of <class 'str'> and finally shows the value in debugging friendly format

record type : <class 'dict'>
fields      : dict_keys(['id', 'source', 'wiki_id', 'snapshot', 'page_id', 'title', 'text'])

id         | str      | 'enwiki_80258789'
source     | str      | 'wikipedia'
wiki_id    | str      | 'enwiki'
snapshot   | str      | '2026-09-01'
page_id    | str      | '80258789'
title      | str      | 'Ehsan Mehmood Khan'
text       | str      | 'is a retired two-star general of the Pakistan Army, who served as the Director General Infantry at the General Headquarters in Rawalpindi.\n\nEducation\nKhan has done his PhD in Peace and Conflict Studies.\n\nMilitary career\nKhan was commissioned into the 34 Punjab Regiment via the 22 OTS Course. 


In [20]:
num_records = 5

with gzip.open(corpus_path, 'rt', encoding='utf-8') as file:

    for i in range(num_records):
        line = file.readline()                        # readline keeps on reading line after line
        record = json.loads(line)

        print("=" * 30)
        print(f"Record : {i + 1}")
        print("=" * 30)

        print("ID     :", record['id'])
        print("Title  :", record['title'])
        print("Text lenght :", len(record['text']))
        print()
        print(record['text'][:500])
        print()


Record : 1
ID     : enwiki_80258789
Title  : Ehsan Mehmood Khan
Text lenght : 1039

is a retired two-star general of the Pakistan Army, who served as the Director General Infantry at the General Headquarters in Rawalpindi.

Education
Khan has done his PhD in Peace and Conflict Studies.

Military career
Khan was commissioned into the 34 Punjab Regiment via the 22 OTS Course. In 2018, Khan was promoted to the rank Major General from Brigadier. As Force Commander Northern Areas, he oversaw military operations and civil-military initiatives in Gilgit-Baltistan. Later he served as D

Record : 2
ID     : enwiki_80258793
Title  : Kathleen Rutherford
Text lenght : 12219

(19 June 1896 – 9 November 1975) was a Scottish-born medical practitioner, philanthropist and humanitarian aid worker. She was for many years a general practitioner (GP) in Harrogate, West Yorkshire, England. She was known locally as and .

Rutherford trained as a physician in Glasgow and Leeds, gained experience as a senior h

### Document level stats

In [22]:
num_documents = 0
total_characters = 0

min_text_length = None
max_text_length = 0

empty_documents = 0

with gzip.open(corpus_path, 'rt', encoding='utf-8') as file:

    for line in file:

        record = json.loads(line)
        text = record['text']

        text_length = len(text)

        num_documents += 1
        total_characters += text_length

        if min_text_length is None or text_length < min_text_length:
            min_text_length = text_length

        if text_length > max_text_length:
            max_text_length = text_length

average_text_length = total_characters/num_documents
print("Documents          :", f"{num_documents:,}")
print("Total characters   :", f"{total_characters:,}")
print("Average text length :", f"{average_text_length:,.2f}")
print("Minimum text length :", f"{min_text_length:,}")
print("Maximum text length :", f"{max_text_length:,}")
print("Empty documents     :", f"{empty_documents:,}")

Documents          : 229,861
Total characters   : 579,738,728
Average text length : 2,522.13
Minimum text length : 200
Maximum text length : 301,416
Empty documents     : 0


In [ ]:
document_lengths = []               # creating an empty list to store each documents length 

with gzip.open(corpus_path, "rt", encoding="utf-8") as file:

    for line in file:

        record = json.loads(line)
        text = record["text"]

        document_lengths.append(len(text))   # directly append in our empty list         


document_lengths.sort()                      # sort ascending order

num_documents = len(document_lengths)

percentiles = [1, 5, 10, 25, 50, 75, 90, 95, 99]    # custom percentiles

print("Document length percentiles")
print()

for percentile in percentiles:

    index = int((percentile / 100) * (num_documents - 1))     # find appropriate list position according to that percentile
    value = document_lengths[index]                           # pull it's value

    print(f"{percentile:>2}th percentile : {value:,} characters")

Document length percentiles

 1th percentile : 212 characters
 5th percentile : 262 characters
10th percentile : 325 characters
25th percentile : 607 characters
50th percentile : 1,374 characters
75th percentile : 2,762 characters
90th percentile : 5,139 characters
95th percentile : 7,741 characters
99th percentile : 18,877 characters


In [ ]:
shortest_records = []
longest_records = []

with gzip.open(corpus_path, 'rt', encoding='utf-8') as file:

    for line in file:

        record = json.loads(line)                       # converts json line into a python dictionary
        text_length = len(record['text'])

        shortest_records.append((text_length, record['title'], record['text']))     # creating a tuple containing a the length, title and full text
        longest_records.append((text_length, record['title'], record['text']))

shortest_records.sort(key=lambda x: x[0])
longest_records.sort(key=lambda x: x[0], reverse=True)